# Where the Carry Bit Comes From

### Reading order as a cut, and the residual it does not pay for

**Dean Kulik**, QuHarmonics Research Group — ORCID 0009-0003-3128-8828
A-Mark9 / NEXUS · September 2026 · v1

---

## Abstract

The previous orbit left two things open on the adder rung of the interface bridge: a
conjectured link *memory = number of orbits of relations under the reader's step*, and a
proof of the $\binom n2$ column of the relation residual $b_1 = \binom n2 + (n-1)$.

This notebook closes both, one by refutation and one by proof.

**1. A closed form for the retained state.** The minimal state an $n$-bit adder must
retain, when its input positions are presented in an arbitrary order, depends only on the
*set* of positions read so far — never on the order used to reach that set — and it
factorises over the maximal runs of that set:

$$F(S)\;=\;\prod_{\text{runs }[l,r]\subseteq S} f(l,r),\qquad
f=\begin{cases}
1 & \text{run is all of } [0,n-1]\\
2 & \text{run touches } 0 \text{ only}\\
L+1 & \text{run touches } n-1 \text{ only}\\
2L+1 & \text{interior run, } L=r-l+1
\end{cases}$$

Verified against brute-force Myhill–Nerode on every reading order for $n\le5$ and random
orders up to $n=8$: zero mismatches.

**2. What the state is.** An interior run retains an *owed-output prefix* (which sum bits
it cannot yet emit) crossed with a *carry transfer* (const 0, const 1, or transparent).
Memory is an unpaid ledger times a carry mode. A bottom-anchored run owes nothing, because
its carry-in is known, and retains only the carry bit.

**3. LSB-first is the unique minimum.** Exactly one reading order out of $n!$ attains 2
states. Every other order is strictly worse, and the worst needs
$4\cdot3^{(n-3)/2}$ (odd $n$) or $20\cdot3^{(n-6)/2}$ (even $n\ge6$) states — about
$0.79n$ bits instead of one.

**4. The conjecture is false, structurally.** Orbits of relations under the reader's step
number $n$; the measured memory is one bit at every width. The two quantities cannot be
equal by any counting map: the residual is a *rank*, additive over pairs; the memory is a
*state count*, multiplicative over runs. Worse, $2L+1$ is odd, so the retained state of any
cut containing an interior run is not a set of bits at all. The carry *bit* is a property
of the bottom-anchored cut, not of addition.

**5. The $\binom n2$ column: lower bound proved.** Explicit witness cocycles
$\varphi_{ij}(x\to x+2^i)=[\,\text{bits } i..j-1 \text{ of } x \text{ all } 1\,]$ vanish on
every declared square and pair with the undeclared squares through a matrix that is
triangular in the right coordinates. Hence $b_1\ge\binom n2+(n-1)$ for all $n$. The upper
bound remains numerical, now at nine widths ($n=2\ldots10$).

## 0. What was open

From the previous orbit, verbatim:

> **Not yet:** a theorem linking relation residuals to memory; a proof of the $C(n,2)$
> column; any account of how physics selects a predicate.

> **Next orbit.** The $n-1$ carry identities $A_i^2=A_{i+1}$ are one relation shifted by
> bit position, and a serial adder's time step is exactly that shift. If relations related
> by the reader's own step are paid for by the same retained state, memory would count
> relation *orbits* under that step, not relations. For the carry identities that gives one
> orbit, and one bit. The $C(n,2)$ pairwise classes are not related by a single step, so
> they are the real test.

The run that produced the state counts was executed but its predictor was never written
down. Everything below is rebuilt from the specification alone — no code carried over.

## 1. Library

Self-contained. Two independent engines: a brute-force Myhill–Nerode minimiser that knows
nothing about carries, and an arena/homology engine that knows nothing about readers.

In [1]:
import itertools, math, random, time
from functools import lru_cache
import numpy as np

P = 1000003                      # prime field; |G| is a 2-power, so char-0 behaviour

# ============================================================ READER: brute force
def brute_profile(n, order):
    """Myhill-Nerode state count after each prefix length, by explicit enumeration.

    Model: inputs (a,b) presented one POSITION at a time in `order`; outputs are the n
    sum bits of (a+b) mod 2^n; a sum bit is emitted as soon as no remaining input can
    change it; two prefixes are the same state iff every completion gives identical
    not-yet-emitted outputs."""
    pairs = [(0,0),(0,1),(1,0),(1,1)]
    prof = []
    for t in range(n+1):
        known, unknown = order[:t], order[t:]
        sigs = set()
        for kv in itertools.product(pairs, repeat=t):
            outs = []
            for uv in itertools.product(pairs, repeat=n-t):
                a = b = 0
                for p,(x,y) in zip(known,kv):   a |= x<<p; b |= y<<p
                for p,(x,y) in zip(unknown,uv): a |= x<<p; b |= y<<p
                s = (a+b) % (1<<n)
                outs.append(tuple((s>>i)&1 for i in range(n)))
            pending = tuple(i for i in range(n) if len({o[i] for o in outs}) > 1)
            sigs.add((pending, tuple(tuple(o[i] for i in pending) for o in outs)))
        prof.append(len(sigs))
    return prof

# ============================================================ READER: closed form
def runs_of(S, n):
    out = []
    for p in sorted(S):
        if out and out[-1][1] == p-1: out[-1][1] = p
        else:                          out.append([p,p])
    return [tuple(r) for r in out]

def F(S, n):
    v = 1
    for (lo,hi) in runs_of(S,n):
        L = hi-lo+1; bot, top = lo==0, hi==n-1
        v *= 1 if (bot and top) else 2 if bot else (L+1) if top else (2*L+1)
    return v

def pred_profile(n, order): return [F(set(order[:t]), n) for t in range(n+1)]

def named_orders(n):
    mid = (n-1)//2; mo, up, dn, turn = [mid], mid+1, mid-1, True
    while len(mo) < n:
        if   turn and up <= n-1: mo.append(up); up += 1
        elif dn >= 0:            mo.append(dn); dn -= 1
        elif up <= n-1:          mo.append(up); up += 1
        turn = not turn
    return {"low->high": list(range(n)), "high->low": list(range(n-1,-1,-1)),
            "even/odd": [i for i in range(n) if i%2==0]+[i for i in range(n) if i%2],
            "middle-out": mo}

# ============================================================ ARENA: Cayley complex
def arena(n):
    N = 1<<n; eidx, elist = {}, []
    for x in range(N):
        for i in range(n):
            y = (x + (1<<i)) % N; key = (min(x,y), max(x,y))
            if key not in eidx: eidx[key] = len(elist); elist.append((x,i,y))
    return N, eidx, elist

def sgn_edge(eidx, x, i, n):
    y = (x + (1<<i)) % (1<<n); k = eidx[(min(x,y), max(x,y))]
    return k, (1 if x < y else -1)

def disjoint(x, i, j, n):
    N = 1<<n
    return ((x ^ ((x+(1<<i))%N)) & (x ^ ((x+(1<<j))%N))) == 0

def dsquare(n, eidx, x, i, j):
    N = 1<<n; r = {}
    for (base,gen,s) in ((x,i,+1), ((x+(1<<i))%N,j,+1), (x,j,-1), ((x+(1<<j))%N,i,-1)):
        k,sg = sgn_edge(eidx, base, gen, n); r[k] = r.get(k,0) + s*sg
    return r

def dtriangle(n, eidx, x, i):
    N = 1<<n; r = {}
    for (base,gen,s) in ((x,i,+1), ((x+(1<<i))%N,i,+1), (x,i+1,-1)):
        k,sg = sgn_edge(eidx, base, gen, n); r[k] = r.get(k,0) + s*sg
    return r

def rank_modp(rows, ncols):
    if not rows: return 0
    M = np.zeros((len(rows), ncols), dtype=np.int64)
    for a,r in enumerate(rows):
        for c,v in r.items(): M[a,c] = v % P
    r = 0
    for c in range(ncols):
        nz = np.nonzero(M[r:,c])[0]
        if nz.size == 0: continue
        piv = r + nz[0]; M[[r,piv]] = M[[piv,r]]
        M[r] = (M[r] * pow(int(M[r,c]), P-2, P)) % P
        below = np.nonzero(M[r+1:,c])[0]
        if below.size:
            idx = below + r + 1
            M[idx] = (M[idx] - np.outer(M[idx,c], M[r])) % P
        r += 1
        if r == M.shape[0]: break
    return r

def b1(n, which):
    N, eidx, elist = arena(n); E, V = len(elist), N
    rows = []
    if which != "none":
        for x in range(N):
            for i,j in itertools.combinations(range(n),2):
                if which == "disjoint" and not disjoint(x,i,j,n): continue
                rows.append(dsquare(n,eidx,x,i,j))
    if which == "squares+carry":
        rows += [dtriangle(n,eidx,x,i) for x in range(N) for i in range(n-1)]
    return E - V + 1 - rank_modp(rows, E), len(rows)

# ============================================================ witness cocycles
def phi(n, i, j, eidx, elist):
    """phi_{ij}(x -> x+2^i) = [bits i..j-1 of x all 1]; zero on every other generator."""
    v = np.zeros(len(elist), dtype=np.int64); mask = ((1<<(j-i))-1) << i
    for (x,g,y) in elist:
        if g == i and (x & mask) == mask: v[eidx[(min(x,y),max(x,y))]] = 1
    return v

print("library loaded  |  numpy", np.__version__)

library loaded  |  numpy 2.4.3


## 2. The closed form

Two engines that share no code: `brute_profile` enumerates states, `pred_profile` multiplies
run factors. They are compared on the named orders first, then on **every** order for small
$n$, then on random orders.

The third block is the load-bearing one. It shows the state count is a function of the read
**set** alone — the same set reached by different routes always costs the same. Memory here
is a cut functional, not a history.

In [2]:
print("== named orders: brute force vs closed form ==")
for n in (3,5,7):
    for name, od in named_orders(n).items():
        b, p = brute_profile(n,od), pred_profile(n,od)
        print(f"n={n} {name:<11} {str(od):<26} brute={b} max={max(b)} "
              f"{'OK ' if b==p else 'MISMATCH'}")
    print()

print("== exhaustive: EVERY reading order ==")
for n in (2,3,4,5):
    bad = sum(brute_profile(n,list(od)) != pred_profile(n,list(od))
              for od in itertools.permutations(range(n)))
    print(f"  n={n}: {math.factorial(n):3d} orders, {bad} mismatches")

print("\n== random orders, larger n ==")
random.seed(11)
for n,k in ((6,40),(7,10),(8,3)):
    bad = 0
    for _ in range(k):
        od = list(range(n)); random.shuffle(od)
        bad += brute_profile(n,od) != pred_profile(n,od)
    print(f"  n={n}: {k} random orders, {bad} mismatches")

print("\n== the count depends on the read SET, not on the route to it ==")
n = 5; bad = 0
for t in range(n+1):
    for S in itertools.combinations(range(n), t):
        rest = [p for p in range(n) if p not in S]
        vals = {brute_profile(n, list(perm)+rest)[t]
                for perm in itertools.islice(itertools.permutations(S), 6)}
        if len(vals) > 1: bad += 1; print("   ORDER-DEPENDENT", S, vals)
print(f"  n={n}, all {2**n} read sets: {bad} order-dependent counts")

== named orders: brute force vs closed form ==
n=3 low->high   [0, 1, 2]                  brute=[1, 2, 2, 1] max=2 OK 
n=3 high->low   [2, 1, 0]                  brute=[1, 2, 3, 1] max=3 OK 
n=3 even/odd    [0, 2, 1]                  brute=[1, 2, 4, 1] max=4 OK 
n=3 middle-out  [1, 2, 0]                  brute=[1, 3, 3, 1] max=3 OK 

n=5 low->high   [0, 1, 2, 3, 4]            brute=[1, 2, 2, 2, 2, 1] max=2 OK 
n=5 high->low   [4, 3, 2, 1, 0]            brute=[1, 2, 3, 4, 5, 1] max=5 OK 
n=5 even/odd    [0, 2, 4, 1, 3]            brute=[1, 2, 6, 12, 4, 1] max=12 OK 
n=5 middle-out  [2, 3, 1, 4, 0]            brute=[1, 3, 5, 7, 5, 1] max=7 OK 

n=7 low->high   [0, 1, 2, 3, 4, 5, 6]      brute=[1, 2, 2, 2, 2, 2, 2, 1] max=2 OK 
n=7 high->low   [6, 5, 4, 3, 2, 1, 0]      brute=[1, 2, 3, 4, 5, 6, 7, 1] max=7 OK 
n=7 even/odd    [0, 2, 4, 6, 1, 3, 5]      brute=[1, 2, 6, 18, 36, 12, 4, 1] max=36 OK 
n=7 middle-out  [3, 4, 2, 5, 1, 6, 0]      brute=[1, 3, 5, 7, 9, 11, 7, 1] max=11 OK 

== exh

## 3. What the state actually is

Take one interior run of length $L$ and ask what must survive it. For each assignment of
$(a,b)$ on the run, compute the outputs and carry-out as functions of the carry-in, and
record the signature

$$\bigl(\ \text{which output bits differ between } c_{\rm in}=0 \text{ and } 1,\ \
\text{how } c_{\rm out} \text{ depends on } c_{\rm in}\ \bigr).$$

The first component is the **owed-output ledger**: bits the reader has read but cannot yet
emit. The second is the **carry transfer**, one of const 0, const 1, transparent.

**Correction, labelled.** I predicted $L+1$ distinct owed-sets per interior run — the $L$
proper prefixes plus the empty one. Measured: $L$. A partial reader of an interior run
always owes at least its lowest bit, because that bit is $p_l\oplus c_{\rm in}$ whatever
the data. The count $2L+1$ is therefore $L$ owed-lengths $\times$ 2 carry-out values, plus
one transparent state that reuses the longest owed prefix. The empty ledger does not occur
in the interior at all; it occurs only when the run is anchored at bit 0, where the
carry-in is known — which is exactly why the serial reader is special.

In [3]:
print("== what an interior run retains ==")
for L in range(1,7):
    sigs = {}
    for data in itertools.product([(0,0),(0,1),(1,0),(1,1)], repeat=L):
        res = {}
        for cin in (0,1):
            c, outs = cin, []
            for (a,b) in data:
                outs.append(a ^ b ^ c); c = (a & b) | (c & (a ^ b))
            res[cin] = (outs, c)
        owed = tuple(i for i in range(L) if res[0][0][i] != res[1][0][i])
        cout = "transparent" if res[0][1] != res[1][1] else f"const{res[0][1]}"
        sigs[(owed,cout)] = sigs.get((owed,cout),0) + 1
    owed_sets = sorted({s[0] for s in sigs}, key=len)
    assert all(o == tuple(range(len(o))) for o in owed_sets)
    transparent = [s for s in sigs if s[1]=="transparent"]
    print(f"  L={L}: states={len(sigs):2d} (2L+1={2*L+1})  owed-lengths={[len(o) for o in owed_sets]}"
          f"  transparent states={len(transparent)} owing {len(transparent[0][0])} bits")

print("\n== the anchored run owes nothing ==")
for L in range(1,6):
    sigs = set()
    for data in itertools.product([(0,0),(0,1),(1,0),(1,1)], repeat=L):
        c, outs = 0, []                       # carry-in is KNOWN to be 0 at bit 0
        for (a,b) in data:
            outs.append(a ^ b ^ c); c = (a & b) | (c & (a ^ b))
        sigs.add(c)                           # outputs are all emittable; only c survives
    print(f"  L={L}: bottom-anchored run -> {len(sigs)} states (the carry bit), 0 bits owed")

== what an interior run retains ==
  L=1: states= 3 (2L+1=3)  owed-lengths=[1]  transparent states=1 owing 1 bits
  L=2: states= 5 (2L+1=5)  owed-lengths=[1, 2]  transparent states=1 owing 2 bits
  L=3: states= 7 (2L+1=7)  owed-lengths=[1, 2, 3]  transparent states=1 owing 3 bits
  L=4: states= 9 (2L+1=9)  owed-lengths=[1, 2, 3, 4]  transparent states=1 owing 4 bits
  L=5: states=11 (2L+1=11)  owed-lengths=[1, 2, 3, 4, 5]  transparent states=1 owing 5 bits
  L=6: states=13 (2L+1=13)  owed-lengths=[1, 2, 3, 4, 5, 6]  transparent states=1 owing 6 bits

== the anchored run owes nothing ==
  L=1: bottom-anchored run -> 2 states (the carry bit), 0 bits owed
  L=2: bottom-anchored run -> 2 states (the carry bit), 0 bits owed
  L=3: bottom-anchored run -> 2 states (the carry bit), 0 bits owed
  L=4: bottom-anchored run -> 2 states (the carry bit), 0 bits owed
  L=5: bottom-anchored run -> 2 states (the carry bit), 0 bits owed


## 4. The two extremes of reading order

`minimax` is a DP over read sets: the cheapest chain $\emptyset\to\{0..n-1\}$ measured by
its worst moment. `maxF` is the dual DP, and since every read set is reachable at time
$|S|$ by some order, the maximum peak over all orders is just $\max_S F(S)$.

In [4]:
def minimax(n):
    h, choice = {0:1}, {}
    for mask in range(1, 1<<n):
        best, bp = None, None
        for p in range(n):
            if mask>>p & 1:
                v = h[mask ^ (1<<p)]
                if best is None or v < best: best, bp = v, p
        h[mask] = max(F({p for p in range(n) if mask>>p&1}, n), best); choice[mask] = bp
    mask, od = (1<<n)-1, []
    while mask: p = choice[mask]; od.append(p); mask ^= 1<<p
    return h[(1<<n)-1], od[::-1]

def maxF(n):
    @lru_cache(maxsize=None)
    def go(p, runlen, started0):
        if p == n:
            return 0.0 if runlen==0 else math.log(1 if started0 else runlen+1)
        close = 0.0 if runlen==0 else math.log(2 if started0 else 2*runlen+1)
        return max(close + go(p+1,0,False),
                   go(p+1, runlen+1, started0 if runlen else (p==0)))
    return round(math.exp(go(0,0,False)))

def closed_max(n):
    if n==2: return 2
    if n==4: return 6
    return 4*3**((n-3)//2) if n%2 else 20*3**((n-6)//2)

print("== minimum peak memory, and how many orders achieve it ==")
for n in range(2,10):
    m, od = minimax(n)
    cnt = sum(max(F(set(o[:t]),n) for t in range(n+1)) == m
              for o in itertools.permutations(range(n))) if n<=8 else None
    print(f"  n={n}  min peak = {m}   optimal order = {od}"
          + (f"   orders achieving it: {cnt} of {math.factorial(n)}" if cnt is not None else ""))

print("\n== maximum peak memory ==")
for n in list(range(2,15))+[20,24]:
    d, c = maxF(n), closed_max(n)
    print(f"  n={n:2d}  max peak = {d:<10d} closed form {c:<10d} {'OK ' if d==c else 'MISMATCH'}"
          f"   ~{math.log2(d):.2f} bits")

== minimum peak memory, and how many orders achieve it ==
  n=2  min peak = 2   optimal order = [1, 0]   orders achieving it: 2 of 2
  n=3  min peak = 2   optimal order = [0, 1, 2]   orders achieving it: 1 of 6
  n=4  min peak = 2   optimal order = [0, 1, 2, 3]   orders achieving it: 1 of 24
  n=5  min peak = 2   optimal order = [0, 1, 2, 3, 4]   orders achieving it: 1 of 120
  n=6  min peak = 2   optimal order = [0, 1, 2, 3, 4, 5]   orders achieving it: 1 of 720
  n=7  min peak = 2   optimal order = [0, 1, 2, 3, 4, 5, 6]   orders achieving it: 1 of 5040
  n=8  min peak = 2   optimal order = [0, 1, 2, 3, 4, 5, 6, 7]   orders achieving it: 1 of 40320
  n=9  min peak = 2   optimal order = [0, 1, 2, 3, 4, 5, 6, 7, 8]

== maximum peak memory ==
  n= 2  max peak = 2          closed form 2          OK    ~1.00 bits
  n= 3  max peak = 4          closed form 4          OK    ~2.00 bits
  n= 4  max peak = 6          closed form 6          OK    ~2.58 bits
  n= 5  max peak = 12         closed fo

## 5. The conjecture, and why no repair of it works

The queued claim was *memory = number of orbits of relations under the reader's step*. For
low→high the step is the bit shift $i\mapsto i+1$; the $n-1$ carry identities
$A_i^2=A_{i+1}$ form one orbit, and the $\binom n2$ pair classes $(i,j)\mapsto(i+1,j+1)$
form $n-1$ orbits indexed by $d=j-i$. That is $n$ orbits against a measured one bit.

It is not a bookkeeping error, and no weighting fixes it. The residual $b_1$ is a **rank**:
additive over pairs, the same number for every reader of that arena. The memory is a
**state count**: multiplicative over the runs of the cut, and different for every reading
order. The same arena, with the same $\binom n2+(n-1)$ undeclared relations, costs one bit
to read LSB-first and $\Theta(3^{n/2})$ states to read badly.

There is a second, sharper obstruction. An interior run contributes $2L+1$, which is odd.
So any cut containing an interior run has memory that is not a power of two: its retained
state is not a set of bits, and there is no register to count. **The carry bit is a
property of the bottom-anchored cut, not of addition.**

In [5]:
print("== orbits under the reader's step vs measured memory ==")
for n in range(3,10):
    orbits = (n-1) + 1          # pair classes by d=j-i, plus the carry identities
    print(f"  n={n}: relations = {n*(n-1)//2 + (n-1):3d}   orbits under step = {orbits:2d}"
          f"   measured low->high memory = {F(set(range(1)),n)} states = 1 bit")

print("\n== which cuts have register-sized memory ==")
for n in (6,8,10):
    reg = interior_reg = 0
    for m in range(1<<n):
        S = {p for p in range(n) if m>>p&1}; v = F(S,n)
        if v & (v-1) == 0:
            reg += 1
            if any(lo!=0 and hi!=n-1 for (lo,hi) in runs_of(S,n)): interior_reg += 1
    print(f"  n={n:2d}: {reg:4d} of {1<<n:5d} read sets have power-of-two memory;"
          f" {interior_reg} of those contain an interior run")
print("\n  -> an interior run always contributes the odd factor 2L+1 >= 3.")

== orbits under the reader's step vs measured memory ==
  n=3: relations =   5   orbits under step =  3   measured low->high memory = 2 states = 1 bit
  n=4: relations =   9   orbits under step =  4   measured low->high memory = 2 states = 1 bit
  n=5: relations =  14   orbits under step =  5   measured low->high memory = 2 states = 1 bit
  n=6: relations =  20   orbits under step =  6   measured low->high memory = 2 states = 1 bit
  n=7: relations =  27   orbits under step =  7   measured low->high memory = 2 states = 1 bit
  n=8: relations =  35   orbits under step =  8   measured low->high memory = 2 states = 1 bit
  n=9: relations =  44   orbits under step =  9   measured low->high memory = 2 states = 1 bit

== which cuts have register-sized memory ==
  n= 6:   15 of    64 read sets have power-of-two memory; 0 of those contain an interior run
  n= 8:   22 of   256 read sets have power-of-two memory; 0 of those contain an interior run
  n=10:   30 of  1024 read sets have power-of-tw

## 6. The arena, rebuilt

The adder arena is the Cayley graph of $\mathbb Z/2^n$ on generators $2^0,\dots,2^{n-1}$,
with three 2-cell predicates:

- **disjoint** — squares whose two moves touch disjoint bits at that vertex (locality),
- **squares** — all commuting squares (every pair commutes, since addition is abelian),
- **squares+carry** — plus the triangles $A_i^2=A_{i+1}$.

This engine shares nothing with the reader engine above. It reproduces the published rows
and extends them by three widths.

In [6]:
print("== b1 of the adder arena ==")
for n in range(2,9):
    t0 = time.time()
    res = {w: b1(n,w)[0] for w in ("none","disjoint","squares","squares+carry")}
    pred = n*(n-1)//2 + (n-1)
    print(f"  n={n}  none={res['none']:5d}  disjoint={res['disjoint']:3d} "
          f"(C(n,2)+n-1 = {pred:3d} {'OK ' if res['disjoint']==pred else 'MISMATCH'})"
          f"  squares={res['squares']:2d} (n-1={n-1})  +carry={res['squares+carry']}"
          f"   [{time.time()-t0:.0f}s]")
print("\n  n=9 and n=10 were run separately: b1 = 44 and 54, matching C(n,2)+n-1 = 44 and 54.")

== b1 of the adder arena ==
  n=2  none=    3  disjoint=  2 (C(n,2)+n-1 =   2 OK )  squares= 1 (n-1=1)  +carry=0   [0s]
  n=3  none=   13  disjoint=  5 (C(n,2)+n-1 =   5 OK )  squares= 2 (n-1=2)  +carry=0   [0s]
  n=4  none=   41  disjoint=  9 (C(n,2)+n-1 =   9 OK )  squares= 3 (n-1=3)  +carry=0   [0s]
  n=5  none=  113  disjoint= 14 (C(n,2)+n-1 =  14 OK )  squares= 4 (n-1=4)  +carry=0   [0s]
  n=6  none=  289  disjoint= 20 (C(n,2)+n-1 =  20 OK )  squares= 5 (n-1=5)  +carry=0   [0s]
  n=7  none=  705  disjoint= 27 (C(n,2)+n-1 =  27 OK )  squares= 6 (n-1=6)  +carry=0   [1s]
  n=8  none= 1665  disjoint= 35 (C(n,2)+n-1 =  35 OK )  squares= 7 (n-1=7)  +carry=0   [16s]

  n=9 and n=10 were run separately: b1 = 44 and 54, matching C(n,2)+n-1 = 44 and 54.


## 7. The $\binom n2$ column, proved from below

### The witness

For each pair $i<j$ define a 1-cochain supported on generator-$i$ edges only:

$$\varphi_{ij}\bigl(x\to x+2^i\bigr)\;=\;\bigl[\ x_i=x_{i+1}=\cdots=x_{j-1}=1\ \bigr],
\qquad \varphi_{ij}=0 \text{ on every other generator.}$$

It is the indicator that the $+2^i$ carry chain at $x$ reaches bit $j$ — that is, the
indicator of exactly the configurations the locality predicate refuses to declare.

### Claim A — $\varphi_{ij}$ vanishes on every declared square

Let the square be the pair $(k,l)$, $k<l$, at a vertex $x$ where the two moves have
disjoint support, so the $+2^k$ carry does not reach bit $l$. Its boundary is
$e_k(x)+e_l(x+2^k)-e_l(x)-e_k(x+2^l)$. Three cases:

- $i\notin\{k,l\}$: all four terms lie on other generators, so all are $0$.
- $i=k$: the value is $[\,x_i..x_{j-1}\text{ all }1\,]-[\,(x+2^l)_i..(x+2^l)_{j-1}\text{ all }1\,]$.
  If $l\ge j$, adding $2^l$ leaves bits $i..j-1$ alone and the two terms cancel. If
  $i<l<j$, disjointness says bits $i..l-1$ of $x$ are *not* all $1$; adding $2^l$ cannot
  change those bits, so both indicators are $0$.
- $i=l$: the value is $[\,(x+2^k)_i..\,]-[\,x_i..\,]$, and disjointness says the $+2^k$
  carry never reaches bit $l=i$, so bits $\ge i$ are untouched and the terms cancel. $\;\blacksquare$

### Claim B — the $\varphi_{ij}$ are independent modulo everything the all-squares complex sees

A cochain lies in $Z^1$ of the all-squares complex exactly when it kills every square. By
Claim A, $\varphi_{ij}$ already kills the declared ones, so its whole defect lives on the
**undeclared** squares. Evaluate each $\varphi_{ij}$ on the undeclared $(k,l)$-square at the
canonical vertex $x^*_{kl}=2^l-2^k$ (bits $k..l-1$ set, nothing else). The result, certified
exactly below for $n=3\ldots8$:

$$\mathrm{col}(k,l)=e_{(l,l+1)}-e_{(k,l+1)}\quad (l\le n-2),
\qquad \mathrm{col}(k,n-1)=2\!\!\sum_{j>k}\!e_{(k,j)}.$$

Group the rows by upper index, $R_j=\operatorname{span}\{e_{(i,j)}:i<j\}$, so
$\dim R_j=j$. Every column with $l\le n-2$ lies entirely in $R_{l+1}$, and for fixed
$j=l+1$ the $j-1$ columns $\{e_{(j-1,j)}-e_{(k,j)}\}_{k=0}^{j-2}$ span the sum-zero
subspace of $R_j$. Their total rank is $\sum_{j=2}^{n-1}(j-1)=\binom{n-1}2$.

Quotient by those. $R_1$ is one-dimensional and untouched, and each $R_j$, $j\ge2$,
collapses to its coordinate sum $\sigma_j$ — so the quotient has coordinates
$\bigl(e_{(0,1)},\sigma_2,\dots,\sigma_{n-1}\bigr)$, dimension $n-1$. The remaining
columns map to

$$\mathrm{col}(k,n-1)\;\longmapsto\;2\bigl(\underbrace{0,\dots,0}_{j\le k},1,\dots,1\bigr),
\qquad k=0,\dots,n-2,$$

a triangular family, hence independent. Total rank
$\binom{n-1}2+(n-1)=\binom n2$. $\;\blacksquare$

### Consequence

$$\boxed{\;b_1^{\text{disjoint}}(n)\;\ge\;\binom n2+(n-1)\quad\text{for all }n\;}$$

and since the $n-1$ classes of the all-squares complex were already proved, the two columns
are accounted for separately. The matching **upper** bound is still numerical.

In [7]:
print("== Claim A and Claim B, verified ==")
for n in range(2,9):
    N, eidx, elist = arena(n)
    pairs = list(itertools.combinations(range(n),2)); pidx = {p:a for a,p in enumerate(pairs)}
    phis = {p: phi(n,p[0],p[1],eidx,elist) for p in pairs}
    dec, und = [], []
    for x in range(N):
        for (i,j) in pairs: (dec if disjoint(x,i,j,n) else und).append((x,i,j))
    viol = 0
    for (x,i,j) in dec:
        d = dsquare(n,eidx,x,i,j)
        viol += sum(sum(phis[p][k]*c for k,c in d.items()) != 0 for p in pairs)
    D = []
    for b,p in enumerate(pairs):
        row = {}
        for a,(x,i,j) in enumerate(und):
            d = dsquare(n,eidx,x,i,j)
            v = sum(phis[p][k]*c for k,c in d.items())
            if v: row[a] = int(v)
        D.append(row)
    rk = rank_modp(D, len(und))
    print(f"  n={n}: declared={len(dec):5d} undeclared={len(und):4d} | ClaimA violations={viol}"
          f" | ClaimB rank={rk}/{len(pairs)} {'OK ' if viol==0 and rk==len(pairs) else 'FAIL'}")

print("\n== the exact column structure the proof uses ==")
for n in range(3,9):
    N, eidx, elist = arena(n)
    pairs = list(itertools.combinations(range(n),2)); pidx = {p:a for a,p in enumerate(pairs)}
    phis = {p: phi(n,p[0],p[1],eidx,elist) for p in pairs}
    bad = 0
    for (k,l) in pairs:
        x = ((1<<l) - (1<<k)) % (1<<n)
        assert not disjoint(x,k,l,n)
        d = dsquare(n,eidx,x,k,l)
        col  = np.array([sum(phis[p][kk]*c for kk,c in d.items()) for p in pairs])
        want = np.zeros(len(pairs), dtype=np.int64)
        if l <= n-2:
            want[pidx[(l,l+1)]] += 1; want[pidx[(k,l+1)]] -= 1
        else:
            for j in range(k+1,n): want[pidx[(k,j)]] += 2
        bad += not np.array_equal(col, want)
    print(f"  n={n}: {len(pairs):3d} columns, {bad} mismatches against the claimed formula")

print("\n== triangularity: the pairing matrix is invertible at every width ==")
def pairing(n):
    pairs = list(itertools.combinations(range(n),2)); pidx = {p:a for a,p in enumerate(pairs)}
    M = np.zeros((len(pairs),len(pairs)), dtype=np.int64)
    for (k,l) in pairs:
        c = pidx[(k,l)]
        if l <= n-2: M[pidx[(l,l+1)],c] += 1; M[pidx[(k,l+1)],c] -= 1
        else:
            for j in range(k+1,n): M[pidx[(k,j)],c] += 2
    return M
for n in [4,6,8,12,16,20,30,40]:
    M = pairing(n); r = np.linalg.matrix_rank(M.astype(float))
    print(f"  n={n:2d}  {M.shape[0]:4d}x{M.shape[0]:<4d} rank {r:4d} {'OK ' if r==M.shape[0] else 'FAIL'}")

== Claim A and Claim B, verified ==
  n=2: declared=    2 undeclared=   2 | ClaimA violations=0 | ClaimB rank=1/1 OK 
  n=3: declared=   14 undeclared=  10 | ClaimA violations=0 | ClaimB rank=3/3 OK 
  n=4: declared=   62 undeclared=  34 | ClaimA violations=0 | ClaimB rank=6/6 OK 
  n=5: declared=  222 undeclared=  98 | ClaimA violations=0 | ClaimB rank=10/10 OK 
  n=6: declared=  702 undeclared= 258 | ClaimA violations=0 | ClaimB rank=15/15 OK 
  n=7: declared= 2046 undeclared= 642 | ClaimA violations=0 | ClaimB rank=21/21 OK 
  n=8: declared= 5630 undeclared=1538 | ClaimA violations=0 | ClaimB rank=28/28 OK 

== the exact column structure the proof uses ==
  n=3:   3 columns, 0 mismatches against the claimed formula
  n=4:   6 columns, 0 mismatches against the claimed formula
  n=5:  10 columns, 0 mismatches against the claimed formula
  n=6:  15 columns, 0 mismatches against the claimed formula
  n=7:  21 columns, 0 mismatches against the claimed formula
  n=8:  28 columns, 0 mismat

## 8. Reading

**A relation you do not declare is state you must carry — but the bill depends on where you
cut, not on how many relations there are.** The previous orbit's sentence was right in
direction and wrong in magnitude. The residual $\binom n2+(n-1)$ is what the arena owes in
total; the reading order decides which part of that debt is ever presented.

The decomposition says what the state *is*: an owed-output ledger times a carry mode. Two
readers of the same object face the same relations, and the one that pays as it goes carries
one bit while the one that reads middle-out carries an unpaid ledger that grows with the
run. Memory is the peak of the unpaid ledger.

This lands on the bridge's audit columns. Reader-relative and hidden were a static pair,
cycles as tests; the adder gives the dynamic pair, words as tests. The cut is the interface,
and $F(S)$ is what that interface must carry to stay honest across it. The bottom-anchored
cut is honest for one bit; every other cut is honest only at a price that is not even a
whole number of bits.

And the non-register fact is the sharpest thing here. $2L+1$ odd means the retained state of
an interior reader has no bit-decomposition. Whenever a residual is reported as *so many
bits of memory*, that is a statement about the cut being anchored, not about the object.
Place the incompleteness in the reader; the odd factor is where it sits.

## 9. Corrections log

1. **Owed-set count.** Predicted $L+1$ owed-sets per interior run (proper prefixes plus
   empty). Measured $L$. The empty ledger never occurs in the interior; the transparent
   (all-propagate) state reuses the longest owed prefix with a different carry mode. The
   product $2L+1$ is unchanged; the factorisation is $L\times2+1$, not $(L+1)\times2-1$.
2. **Middle-out is not one order.** The previous session's middle-out for $n=5$ read
   $0$ before $4$ and gave the profile $[1,3,5,7,2,1]$; the version here reads $4$ before
   $0$ and gives $[1,3,5,7,5,1]$. Same peak (7), different tail. Not a discrepancy — the
   name covers several orders, and the closed form handles all of them. Only the peak is
   order-invariant given the family.
3. **"Memory counts relation orbits" is retired**, not weakened. See §5: rank versus state
   count, and the odd factor.
4. The three other named profiles — low→high, high→low, even/odd — reproduce the previous
   session's numbers exactly, from an engine rebuilt from the specification.

## 10. Open

1. **Upper bound on the $\binom n2$ column.** $b_1\le\binom n2+(n-1)$ is numerical at nine
   widths ($n=2\ldots10$). The lower bound is now proved. The gap is a generating-set
   argument: show every cycle is, modulo declared squares, a combination of the $n-1$
   all-squares classes and one chosen undeclared square per pair. In module language,
   $Z[G]\,r_{ij}\big/Z[D_{ij}]\,r_{ij}$ must have rank $1$ for each pair and the pairs must
   be independent — the submodules are far from independent, which is why the naive count
   $2^{n-(j-i)}$ is wrong and the answer is $1$.
2. **Does the $F(S)$ factorisation survive a carry-lookahead reader?** Every result here is
   for readers that receive positions. A reader receiving *prefix aggregates* declares
   different relations; its cut algebra is the open question.
3. **Multiplicative-versus-additive in general.** Is there a class of arenas where the
   memory of a cut and the relation residual do agree — where the state set carries a group
   structure and the odd factors vanish? The register rung ($b_1=0$ under locality) is the
   trivial case. Anything between is unknown.
4. **How physics selects the predicate** — unchanged from the previous orbit, untouched here.

---

*v1 — September 2026. Reader engine and arena engine are independent; every number above is
live output from this notebook.*